## BEFORE EVERY SESSION -- REQUIRED

**Restart the Jupyter kernel before running any new session.**

Kernel > Restart Kernel and Clear All Outputs

Failure to restart will cause stale state from the previous session
to contaminate the new session. Results will be incorrect.

This is not optional.

# The Creative Prism
## Phase 1 — Full Studio Workflow
**v2.0 — Adaptive Discovery, Routing, Team Configuration**

```
Stage 0    Routing              ← Director classifies: STUDIO / DIRECT / OUT_OF_SCOPE
Stage 1    Adaptive Discovery   ← Director-controlled loop, 1-6 turns, extraction toolkit
Stage 2    Reframing            ← Director proposes reframing, PIL confirms
Stage 3    Creative Brief       ← JSON format, validation gate
Stage 3a   Team Configuration   ← Director tunes Creative Team traits for this problem (NEW)
Stage 3b   Open Questions       ← Director surfaces to PIL before ideation
Stage 4a   Creator Pass 1       ← 3 directions, banded Verbalized Sampling
Stage 4b   Critic Pass 1        ← token-efficient evaluation, RESEARCH_REQUEST
Stage 4c   Researcher           ← responds to requests + autonomous contribution
Stage 4d   Creator Pass 2       ← refines directions based on research
Stage 4e   Critic Pass 2        ← final evaluation + synthesis → Director
Stage 5    Candidate Directions
Stage 6    Director Review      ← JSON format, reads from latest ideation cycle
Stage 6a   Iteration Loop       ← re-runs full Creative Team if ITERATE
Stage 7    Presentation
Stage 8    User Reaction
Stage 8b   Depth Extraction
Stage 9    Final Synthesis
```

**v1.5 changes:**
- Stage 0: Director routing — classifies request before engaging studio
- Stage 1: Adaptive discovery loop — Director controls turn count (1-6), deploys extraction toolkit
- Stage 2: Director proposes reframing to PIL for confirmation before briefing
- Stage 3a: Director configures Creative Team personality traits for this specific problem
- DIRECTOR_MODEL separate from SESSION_MODEL — Sonnet for Director, Haiku for team
- All PIL interactions are live input — no simulated personas
- Full conversation history stored in Studio Brief Document


---
## Cell 1 — Load Phase 0 Foundation

In [ ]:
import re
import json

from engine import (
    client,
    create_blackboard,
    scribe_log,
    build_prompt,
    call_role,
    save_session,
    verify_engine,
    create_brief_doc,
    update_brief_doc,
    read_brief_doc,
    load_traits_matrix,
    weight_to_band,
    build_trait_profile,
    get_tunable_traits,
    validate_adjustments,
    validate_stage,
    generate_baseline,
    assemble_evaluation_payload,
    DEFAULT_MODEL,
)

engine_ready = verify_engine()
if not engine_ready:
    raise RuntimeError("Engine verification failed.")

traits_matrix = load_traits_matrix()
print(f"\nTraits matrix: {len(traits_matrix)} active traits")
print("Engine ready")

---
## Cell 2 — Session Configuration

All PIL interactions are live input.

In [ ]:
# -- SESSION CONFIGURATION -------------------------------------------------

DIRECTOR_MODEL = "claude-sonnet-4-20250514"
SESSION_MODEL = "claude-haiku-4-5-20251001"

# Session-level trait adjustments -- populated by Director in Stage 3a
session_adjustments = {"creator": {}, "critic": {}, "researcher": {}}

# Built trait profiles -- populated after Stage 3a
trait_profiles = {"creator": "", "critic": "", "researcher": ""}

print(f"Director model : {DIRECTOR_MODEL}")
print(f"Session model  : {SESSION_MODEL}")

---
## Cell 3 — Stage 0: Routing

The Director's first decision: is this a creative challenge for the studio,
a simple request to handle directly, or something out of scope entirely?

STUDIO → enter adaptive discovery.
DIRECT → Director responds in one turn, session ends.
OUT_OF_SCOPE → warm redirect, session ends.

In [ ]:
# -- STAGE 0: ROUTING ------------------------------------------------------

print("=" * 60)
print("STAGE 0 -- ROUTING")
print("=" * 60)

initial_prompt = input(
    "Welcome to The Creative Prism. What would you like to explore today?\n> "
).strip()
# -- Baseline generation (Control B) -----------------------------------
# Zero-shot response to raw prompt. Stored for evaluation comparison.
# Uses SESSION_MODEL — same capability as Creative Team, architecture
# is the isolated variable. Called once, before any agents run.
baseline_response = generate_baseline(initial_prompt, model=SESSION_MODEL)
print("  Baseline generated and stored.")

# -- Create session --------------------------------------------------------
blackboard = create_blackboard(
    initial_prompt, system_version="2.1", director_model=DIRECTOR_MODEL
)
session_id = blackboard["session_metadata"]["session_id"]
brief_doc_path = create_brief_doc(session_id, initial_prompt)
scribe_log(
    blackboard, "SYSTEM", "session_start", f"Session initiated: '{initial_prompt}'"
)

# -- Director routing call -------------------------------------------------
routing_task = (
    "A person has just arrived at The Creative Prism with this request:\n\n"
    f'"{initial_prompt}"\n\n'
    "Determine the right route for this request.\n\n"
    "STUDIO -- A creative challenge that benefits from structured exploration: "
    "brand development, strategic thinking, creative direction, problem reframing, "
    "experience design, naming, positioning, identity work, or any challenge "
    "where multiple perspectives would help. Most requests should route here.\n\n"
    "DIRECT -- A simple, factual, or straightforward request you can answer "
    "directly in one response. A quick opinion, a factual answer, a simple "
    "recommendation.\n\n"
    "OUT_OF_SCOPE -- Not what The Creative Prism is designed for: medical "
    "information, legal advice, homework help, code debugging, schedules, "
    "general knowledge queries, translation, or anything that is not a "
    "creative or strategic challenge.\n\n"
    "Return ONLY a JSON object -- no preamble, no markdown fences:\n\n"
    "{\n"
    '  "route": "STUDIO",\n'
    '  "message": "your response to the person"\n'
    "}\n\n"
    "If STUDIO: Warmly acknowledge what they have brought. Demonstrate that "
    "you understood it. Then ask ONE targeted opening question to begin "
    "understanding the person behind the request. The person may not be "
    "experienced with AI tools -- meet them where they are. Their initial "
    "prompt may be vague or incomplete and that is completely normal.\n\n"
    "If DIRECT: Your message is your complete, helpful response.\n\n"
    "If OUT_OF_SCOPE: Warmly explain what The Creative Prism is designed "
    "for -- creative exploration, brand development, strategic thinking, "
    "problem reframing -- and invite them to bring a creative challenge. "
    "Two to three sentences. Confident and redirecting, not apologetic."
)

routing_response = call_role(
    role="director",
    user_message=routing_task,
    context=initial_prompt,
    blackboard=blackboard,
    model=DIRECTOR_MODEL,
    max_tokens=400,
    brief_doc="",
)

# -- Parse routing response ------------------------------------------------
try:
    clean = routing_response.strip()
    if clean.startswith("```"):
        clean = clean.split("```")[1]
        if clean.startswith("json"):
            clean = clean[4:]
    routing = json.loads(clean.strip())
    session_route = routing.get("route", "STUDIO").upper()
    director_message = routing.get("message", "")
except (json.JSONDecodeError, ValueError):
    session_route = "STUDIO"
    director_message = routing_response.strip()

print(f"\nRoute: {session_route}\n")
print(f"Director: {director_message}")

# -- Handle non-studio routes ---------------------------------------------
if session_route == "OUT_OF_SCOPE":
    scribe_log(
        blackboard,
        "DIRECTOR",
        "routing_out_of_scope",
        f"Request routed OUT_OF_SCOPE: '{initial_prompt[:60]}'",
    )
    update_brief_doc(
        session_id, "DIRECTOR", "ROUTING", f"Route: OUT_OF_SCOPE\n\n{director_message}"
    )
    save_session(blackboard)
    print("\n" + "=" * 60)
    print("SESSION COMPLETE -- out of scope request handled gracefully.")
    print("No further cells need to be run.")
    print("=" * 60)

elif session_route == "DIRECT":
    scribe_log(
        blackboard,
        "DIRECTOR",
        "routing_direct",
        f"Request routed DIRECT: '{initial_prompt[:60]}'",
    )
    update_brief_doc(
        session_id, "DIRECTOR", "ROUTING", f"Route: DIRECT\n\n{director_message}"
    )
    save_session(blackboard)
    print("\n" + "=" * 60)
    print("SESSION COMPLETE -- direct response delivered.")
    print("No further cells need to be run.")
    print("=" * 60)

else:
    scribe_log(
        blackboard,
        "DIRECTOR",
        "routing_studio",
        "Request routed to STUDIO. Entering adaptive discovery.",
    )
    print(f"\nRouted to studio -- entering adaptive discovery")
    print(f"Session ID: {session_id[:8]}...")
    print(f"Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

---
## Cell 4 — Stage 1: Adaptive Discovery

Director-controlled discovery loop. One to six turns.
The Director decides when it has enough signal — not the notebook.

Each turn: Director speaks → PIL responds → Director assesses signal.
The Director can deploy extraction techniques (sacrificial concepts,
forced choices, fill-in-the-blank, etc.) when signal is weak.

All PIL interactions are live input.

In [ ]:
# -- STAGE 1: ADAPTIVE DISCOVERY -------------------------------------------

print("=" * 60)
print("STAGE 1 -- ADAPTIVE DISCOVERY")
print("=" * 60)

if session_route != "STUDIO":
    print("Skipped -- session was routed as", session_route)
else:

    MAX_DISCOVERY_TURNS = 6

    # -- Discovery state ---------------------------------------------------
    conversation_history = []
    all_assessments = []
    discovery_status = "CONTINUE"
    current_signals = []
    current_missing = []

    # The routing call already produced the Director's first message.
    conversation_history.append({"role": "director", "content": director_message})

    print(f"\n-- Turn 1 -------------------------------------------------")
    print(f"Director: {director_message}\n")

    # -- Collect first PIL response ----------------------------------------
    pil_response = input("> ").strip()
    conversation_history.append({"role": "pil", "content": pil_response})

    # -- Discovery loop ----------------------------------------------------
    turn = 1

    while discovery_status == "CONTINUE" and turn < MAX_DISCOVERY_TURNS:
        turn += 1

        # Format conversation for the Director
        history_formatted = "\n".join(
            [
                ("DIRECTOR: " if e["role"] == "director" else "PERSON: ") + e["content"]
                for e in conversation_history
            ]
        )

        # Ceiling pressure on final turn
        ceiling_note = ""
        if turn == MAX_DISCOVERY_TURNS:
            ceiling_note = (
                f"\n\nThis is turn {turn} of {MAX_DISCOVERY_TURNS} -- your last turn. "
                "Set status to READY and work with the signal you have."
            )

        discovery_task = (
            "You are the Director of The Creative Prism, in a discovery "
            "conversation with a person who has brought a creative challenge.\n\n"
            f"CONVERSATION SO FAR:\n{history_formatted}\n\n"
            f"TURN: {turn} of {MAX_DISCOVERY_TURNS}{ceiling_note}\n\n"
            "YOUR EXTRACTION TOOLKIT (use when signal is weak or vague):\n"
            "- Forced Choice: Does this feel more like A or B?\n"
            "- Elimination: Which of these feels least right?\n"
            "- Extremes Test: Push concept to an extreme, observe comfort\n"
            "- Reaction Snapshot: Quick gut reaction, before you think about it?\n"
            "- Fill in the Blank: This should feel more like ___ than ___\n"
            "- Analogy Probe: If this were a place/object/experience, what would it be?\n"
            "- Constraint Introduction: If you had to explain in one sentence, what matters most?\n"
            "- Signal Amplification: Reflect what you heard slightly amplified, invite correction\n"
            "- Micro-Ranking: Rank these 1-3 on instinct: X / Y / Z\n"
            "- Sacrificial Concept: Present a deliberately wrong/strange idea -- "
            "the value is in what the person pushes back against and why\n"
            "- Category Transplant: Propose the solution as if it belonged to "
            "a different industry entirely\n\n"
            "GUIDANCE:\n"
            "- After turn 2, if the person gives broad or vague answers, DEPLOY a specific\n"
            "  extraction technique. Do not ask another open-ended question. Use a forced\n"
            "  choice, fill-in-the-blank, sacrificial concept, or analogy probe instead.\n"
            "- Offer specific alternatives rather than asking how they see things.\n"
            "- The person may not be experienced with AI. Meet them where they are.\n"
            "- Vague or incomplete answers are normal -- your toolkit handles this.\n"
            "- Acknowledge what they said before your next question. Brief, genuine.\n"
            "- ONE question or probe per turn. Never stack multiple questions.\n"
            "- Use extraction techniques when standard questions produce vague signal.\n"
            "- You are trying to understand the PERSON, not just the problem.\n"
            "- Keep it conversational -- a smart, engaged collaborator.\n\n"
            "WHAT YOU NEED FOR A CREATIVE BRIEF:\n"
            "- The challenge: what they are actually trying to solve\n"
            "- The context: who it is for, where it lives, what the situation is\n"
            "- What success looks like: desired outcome\n"
            "- What to avoid: boundaries, anti-patterns, negative space\n"
            "- The person: what they value beneath what they have said\n\n"
            "CRITICAL RULE FOR STATUS:\n"
            "- If you set status to READY, your message must be a CLOSING STATEMENT,\n"
            "  not a question. Summarize what you have learned and signal that you are\n"
            "  about to move to the next phase. Do NOT ask a question and set READY.\n"
            "- If you still have a question to ask, set status to CONTINUE.\n\n"
            "RESPOND IN TWO PARTS:\n\n"
            "PART 1: Your message to the person. Conversational, warm, one "
            "question or probe. Acknowledge what they said first.\n\n"
            "---SIGNAL_ASSESSMENT---\n\n"
            "PART 2: A JSON signal assessment (the person will NOT see this):\n"
            "{\n"
            '  "signals_gathered": ["list each distinct signal established so far"],\n'
            '  "signals_missing": ["what you still need to understand"],\n'
            '  "technique_used": "none | forced_choice | elimination | extremes_test | '
            "reaction_snapshot | fill_blank | analogy_probe | constraint_intro | "
            'signal_amplification | micro_ranking | sacrificial_concept | category_transplant",\n'
            '  "status": "CONTINUE | READY",\n'
            '  "reason": "one sentence -- why you need more or why you have enough"\n'
            "}"
        )

        director_response = call_role(
            role="director",
            user_message=discovery_task,
            context=initial_prompt,
            blackboard=blackboard,
            model=DIRECTOR_MODEL,
            max_tokens=600,
            brief_doc=read_brief_doc(session_id),
        )

        # -- Parse the two-part response -----------------------------------
        if "---SIGNAL_ASSESSMENT---" in director_response:
            parts = director_response.split("---SIGNAL_ASSESSMENT---", 1)
            director_message = parts[0].strip()
            assessment_raw = parts[1].strip()
        else:
            director_message = director_response.strip()
            assessment_raw = json.dumps(
                {
                    "signals_gathered": current_signals,
                    "signals_missing": ["assessment delimiter not found"],
                    "technique_used": "none",
                    "status": "CONTINUE" if turn < MAX_DISCOVERY_TURNS else "READY",
                    "reason": "Signal assessment delimiter not found in response",
                }
            )

        # Parse signal assessment JSON
        try:
            clean_a = assessment_raw.strip()
            if clean_a.startswith("```"):
                clean_a = clean_a.split("```")[1]
                if clean_a.startswith("json"):
                    clean_a = clean_a[4:]
            # Find closing brace to handle trailing text
            brace_depth = 0
            json_end = 0
            for ci, ch in enumerate(clean_a):
                if ch == "{":
                    brace_depth += 1
                elif ch == "}":
                    brace_depth -= 1
                    if brace_depth == 0:
                        json_end = ci + 1
                        break
            if json_end > 0:
                clean_a = clean_a[:json_end]
            assessment = json.loads(clean_a.strip())
        except (json.JSONDecodeError, ValueError, IndexError):
            assessment = {
                "signals_gathered": current_signals,
                "signals_missing": ["assessment parsing failed"],
                "technique_used": "unknown",
                "status": "CONTINUE" if turn < MAX_DISCOVERY_TURNS else "READY",
                "reason": "JSON parsing failed",
            }

        discovery_status = assessment.get("status", "CONTINUE").upper()
        current_signals = assessment.get("signals_gathered", current_signals)
        current_missing = assessment.get("signals_missing", [])
        technique = assessment.get("technique_used", "none")
        reason = assessment.get("reason", "")
        all_assessments.append(assessment)

        conversation_history.append({"role": "director", "content": director_message})

        # -- Display -------------------------------------------------------
        print(f"-- Turn {turn} -------------------------------------------------")
        print(f"Director: {director_message}\n")
        if technique and technique != "none":
            print(f"  [technique: {technique}]")
        print(f"  [status: {discovery_status} | {reason}]")
        print()

        if discovery_status == "READY":
            # Safety net: if Director asked a question while setting READY,
            # collect the response before exiting
            if director_message.rstrip().endswith("?"):
                print("  [Director asked a question with READY — collecting response]")
                pil_response = input("> ").strip()
                conversation_history.append({"role": "pil", "content": pil_response})
            break

        # -- Collect PIL response ------------------------------------------
        pil_response = input("> ").strip()
        conversation_history.append({"role": "pil", "content": pil_response})

    # -- Store discovery in Blackboard and Brief Document ------------------
    blackboard["discovery"]["orientation_summary"] = initial_prompt
    blackboard["discovery"]["context_insights"] = current_signals
    blackboard["discovery"]["preferences"] = [
        s
        for s in current_signals
        if any(
            kw in s.lower()
            for kw in ["avoid", "prefer", "want", "don't", "hate", "love", "feel"]
        )
    ]

    # Check if sacrificial concept was used
    for idx, a in enumerate(all_assessments):
        if a.get("technique_used") == "sacrificial_concept":
            dir_idx = (idx + 1) * 2
            if dir_idx < len(conversation_history):
                probe = conversation_history[dir_idx].get("content", "")
                resp_idx = dir_idx + 1
                resp = (
                    conversation_history[resp_idx].get("content", "")
                    if resp_idx < len(conversation_history)
                    else ""
                )
                blackboard["discovery"]["sacrificial_exploration"][
                    "probe_prompt"
                ] = probe
                blackboard["discovery"]["sacrificial_exploration"][
                    "user_response"
                ] = resp
                blackboard["discovery"]["sacrificial_exploration"][
                    "insight_extracted"
                ] = f"Sacrificial concept deployed at turn {idx+2}. PIL reaction recorded."
            break

    # Record techniques used
    techniques_used = [
        a.get("technique_used", "none")
        for a in all_assessments
        if a.get("technique_used", "none") != "none"
    ]
    blackboard["discovery"]["exploratory_prompts"] = techniques_used

    # Store full conversation in Studio Brief Document
    discovery_log = "\n\n".join(
        [
            ("**Director:** " if e["role"] == "director" else "**Person:** ")
            + e["content"]
            for e in conversation_history
        ]
    )
    update_brief_doc(session_id, "DIRECTOR", "DISCOVERY CONVERSATION", discovery_log)

    if current_signals:
        signals_section = "\n".join(f"- {s}" for s in current_signals)
        update_brief_doc(session_id, "DIRECTOR", "SIGNALS GATHERED", signals_section)

    if current_missing:
        missing_section = "\n".join(f"- {m}" for m in current_missing)
        update_brief_doc(session_id, "DIRECTOR", "OPEN SIGNAL GAPS", missing_section)

    scribe_log(
        blackboard,
        "DIRECTOR",
        "discovery_complete",
        f"Adaptive discovery complete. {turn} turns. "
        f"{len(current_signals)} signals gathered. "
        f"Techniques: {techniques_used if techniques_used else 'none'}.",
    )

    print(f"\nDiscovery complete -- {turn} turns")
    print(f"Signals gathered: {len(current_signals)}")
    if techniques_used:
        print(f"Techniques used: {', '.join(techniques_used)}")
    if current_missing:
        print(f"Still missing: {', '.join(current_missing[:3])}")
    print(f"Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

---
## Cell 5 — Stage 2 & 3: Reframing + Creative Brief

The Director proposes a reframing of the challenge to the PIL for confirmation.
This is the moment where the PIL sees whether the Director understood them.
After confirmation, the Director produces the structured creative brief.

The reframing step closes the discovery loop by
demonstrating understanding before the Creative Team is engaged.

In [ ]:
# -- STAGE 2: REFRAMING ----------------------------------------------------
# -- STAGE 3: CREATIVE BRIEF -----------------------------------------------

print("=" * 60)
print("STAGE 2 -- REFRAMING")
print("STAGE 3 -- CREATIVE BRIEF")
print("=" * 60)

if session_route != "STUDIO":
    print("Skipped -- session was routed as", session_route)
else:

    # -- Build discovery summary from conversation -------------------------
    discovery_log_plain = "\n".join(
        [
            ("Director: " if e["role"] == "director" else "Person: ") + e["content"]
            for e in conversation_history
        ]
    )

    signals_summary = (
        "\n".join(f"- {s}" for s in current_signals)
        if current_signals
        else "None captured"
    )
    missing_summary = (
        "\n".join(f"- {m}" for m in current_missing) if current_missing else "None"
    )

    # -- Director proposes reframing ---------------------------------------
    reframe_task = (
        "You have just completed a discovery conversation. Now propose a "
        "reframing of the person's challenge.\n\n"
        f"DISCOVERY CONVERSATION:\n{discovery_log_plain}\n\n"
        f"SIGNALS GATHERED:\n{signals_summary}\n\n"
        f"GAPS REMAINING:\n{missing_summary}\n\n"
        "Your reframing should:\n"
        "- Show the person you understood not just what they said, but what they meant\n"
        "- Restate the challenge in a way that is sharper than how they put it\n"
        "- Surface any insight from the conversation they may not have seen themselves\n"
        "- Be 3-5 sentences -- clear, confident, specific to this person\n\n"
        "End by asking the person to confirm this is right, or to adjust anything "
        "that is off. This is the moment where they see whether you understood them."
    )

    reframe_response = call_role(
        role="director",
        user_message=reframe_task,
        context=initial_prompt,
        blackboard=blackboard,
        model=DIRECTOR_MODEL,
        max_tokens=400,
        brief_doc=read_brief_doc(session_id),
    )

    print("-- DIRECTOR REFRAMING ------------------------------------------")
    print(reframe_response)
    print()

    # -- PIL confirms or adjusts -------------------------------------------
    pil_confirmation = input("> ").strip()

    # Store reframing exchange
    update_brief_doc(
        session_id,
        "DIRECTOR",
        "REFRAMING",
        f"**Director:**\n{reframe_response}\n\n"
        f"**Person confirmed:**\n{pil_confirmation}",
    )
    scribe_log(
        blackboard,
        "DIRECTOR",
        "reframing_confirmed",
        "Reframing proposed and confirmed by PIL.",
    )

    blackboard["discovery"]["desired_outcome"] = reframe_response

    # -- Director synthesizes creative brief -------------------------------
    brief_task = (
        "Based on this confirmed understanding, produce a structured creative brief.\n\n"
        f"DISCOVERY CONVERSATION:\n{discovery_log_plain}\n\n"
        f"REFRAMING (confirmed by person):\n{reframe_response}\n\n"
        f"PERSON'S CONFIRMATION:\n{pil_confirmation}\n\n"
        f"SIGNALS GATHERED:\n{signals_summary}\n\n"
        "Return ONLY a JSON object -- no preamble, no markdown fences:\n\n"
        "{\n"
        '  "challenge": "one clear sentence -- the actual challenge",\n'
        '  "context": "2-3 sentences about situation, audience, and what matters",\n'
        '  "desired_result": "what success looks like -- specific to what was revealed",\n'
        '  "constraints": ["constraint 1", "constraint 2"],\n'
        '  "open_questions": ["question 1", "question 2"]\n'
        "}\n\n"
        "Be specific. This brief becomes the Creative Team's working document. "
        "Include what the person explicitly said AND what their reactions revealed."
    )

    brief_response = call_role(
        role="director",
        user_message=brief_task,
        context=initial_prompt,
        blackboard=blackboard,
        model=DIRECTOR_MODEL,
        max_tokens=600,
        brief_doc=read_brief_doc(session_id),
    )

    # -- Parse brief -- JSON with validation gate --------------------------
    brief = blackboard["creative_brief"]

    try:
        clean = brief_response.strip()
        if clean.startswith("```"):
            clean = clean.split("```")[1]
            if clean.startswith("json"):
                clean = clean[4:]
        parsed = json.loads(clean.strip())

        brief["challenge"] = parsed.get("challenge", "")
        brief["context"] = parsed.get("context", "")
        brief["desired_result"] = parsed.get("desired_result", "")
        brief["constraints"] = parsed.get("constraints", [])
        brief["open_questions"] = parsed.get("open_questions", [])

        missing = [
            f
            for f in ["challenge", "context", "desired_result"]
            if not brief.get(f, "").strip()
        ]
        if missing:
            raise ValueError(f"Brief missing required fields: {missing}")

        print("Brief parsed successfully")

    except (json.JSONDecodeError, ValueError) as e:
        print(f"Brief parsing failed: {e}")
        print("Raw response:")
        print(brief_response)
        raise RuntimeError(
            "Brief parsing failed -- session cannot continue with an incomplete brief.\n"
            "Review the raw response above and re-run this cell."
        )

    # -- Store and display -------------------------------------------------
    update_brief_doc(
        session_id,
        "DIRECTOR",
        "CREATIVE BRIEF",
        f"**Challenge:** {brief['challenge']}\n\n"
        f"**Context:** {brief['context']}\n\n"
        f"**Desired result:** {brief['desired_result']}\n\n"
        f"**Constraints:** {', '.join(brief['constraints'])}\n\n"
        f"**Open questions:** {', '.join(brief['open_questions'])}",
    )

    scribe_log(
        blackboard,
        "DIRECTOR",
        "brief_synthesized",
        f"Creative brief produced. Challenge: '{brief['challenge'][:60]}...'",
    )
    validate_stage(blackboard, "brief")

    print()
    print(f"Challenge      : {brief['challenge']}")
    print(f"Context        : {brief['context'][:80]}...")
    print(f"Desired result : {brief['desired_result'][:80]}...")
    print(f"Constraints    : {brief['constraints']}")
    print(f"Open questions : {len(brief['open_questions'])}")
    print()
    print(f"Creative brief complete")
    print(f"Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

---
## Cell 5a — Stage 3a: Team Configuration

The Director configures the Creative Team's personality traits
for this specific problem. Adjusts 9 core trait weights and writes
a short team brief for each role (Creator, Critic, Researcher).

This is invisible to the PIL — it is the Director selecting the
right team for the job, exactly as a Creative Director would in
a real agency. The team brief is written to the Studio Brief Document
so every role reads it before acting.

In [ ]:
# -- STAGE 3a: TEAM CONFIGURATION ------------------------------------------

print("=" * 60)
print("STAGE 3a -- TEAM CONFIGURATION")
print("=" * 60)

if session_route != "STUDIO":
    print("Skipped")
else:
    tunable_summary = ""
    for role_name in ["creator", "critic", "researcher"]:
        tunable = get_tunable_traits(role_name, traits_matrix)
        tunable_summary += "\n" + role_name.upper() + " tunable traits:\n"
        for t in tunable:
            tunable_summary += (
                "  "
                + t["trait"]
                + " (base: "
                + f"{t['base']:.2f}, range: {t['min']:.2f}-{t['max']:.2f})"
                f" -- {t['description']}\n"
            )

    config_task = (
        "You are the Director. The brief is complete. Configure the Creative "
        "Team for this specific problem.\n\n"
        f"BRIEF:\n"
        f"Challenge: {brief['challenge']}\n"
        f"Context: {brief['context']}\n"
        f"Constraints: {', '.join(brief['constraints'])}\n\n"
        "You are selecting the right team personality for this job -- like a "
        "Creative Director choosing which creatives to assign.\n\n"
        f"TUNABLE TRAITS (only adjust 3-8 that matter most):\n{tunable_summary}\n\n"
        "Return ONLY JSON -- no preamble:\n"
        "{\n"
        '  "creator_adjustments": {"trait_name": value, ...},\n'
        '  "critic_adjustments": {"trait_name": value, ...},\n'
        '  "researcher_adjustments": {"trait_name": value, ...},\n'
        '  "rationale": "2-3 sentences"\n'
        "}\n\n"
        "Only list traits you want to CHANGE from base. Unmentioned traits stay "
        "at their base values. Values must be within the allowed range."
    )

    config_response = call_role(
        role="director",
        user_message=config_task,
        context=brief["challenge"],
        blackboard=blackboard,
        model=DIRECTOR_MODEL,
        max_tokens=600,
        brief_doc=read_brief_doc(session_id),
    )

    try:
        clean = config_response.strip()
        if clean.startswith("```"):
            clean = clean.split("```")[1]
            if clean.startswith("json"):
                clean = clean[4:]
        brace_depth = 0
        json_end = 0
        for ci, ch in enumerate(clean):
            if ch == "{":
                brace_depth += 1
            elif ch == "}":
                brace_depth -= 1
                if brace_depth == 0:
                    json_end = ci + 1
                    break
        if json_end > 0:
            clean = clean[:json_end]
        config = json.loads(clean.strip())

        for role_name in ["creator", "critic", "researcher"]:
            raw = config.get(role_name + "_adjustments", {})
            validated, warnings = validate_adjustments(role_name, raw, traits_matrix)
            session_adjustments[role_name] = validated
            for w in warnings:
                print(f"  Warning: {w}")

        for role_name in ["creator", "critic", "researcher"]:
            trait_profiles[role_name] = build_trait_profile(
                role_name, session_adjustments[role_name], traits_matrix
            )

        rationale = config.get("rationale", "")
        team_brief = "**Director reasoning:** " + rationale + "\n"
        for role_name in ["creator", "critic", "researcher"]:
            adj = session_adjustments[role_name]
            if adj:
                adj_str = ", ".join(f"{k}: {v:.2f}" for k, v in adj.items())
                team_brief += "\n**" + role_name.upper() + " adjustments:** " + adj_str

        update_brief_doc(session_id, "DIRECTOR", "TEAM CONFIGURATION", team_brief)
        scribe_log(
            blackboard,
            "DIRECTOR",
            "team_configured",
            f"Team configured. {rationale[:80]}...",
        )

        print(f"Team configured.\n\nRationale: {rationale}\n")
        for role_name in ["creator", "critic", "researcher"]:
            adj = session_adjustments[role_name]
            n = len(adj)
            print(
                f"{role_name.upper()}: {n} traits adjusted"
                + (f" -- {adj}" if adj else "")
            )

    except (json.JSONDecodeError, ValueError) as e:
        print(f"Team config parsing failed: {e}. Using default traits.")
        for role_name in ["creator", "critic", "researcher"]:
            trait_profiles[role_name] = build_trait_profile(
                role_name, {}, traits_matrix
            )

---
## Cell 4b — Stage 3b: Open Questions to PIL

Director surfaces open questions from the brief to the PIL before ideation begins.
Answers shape the brief and the Studio Brief Document.
PIL responds to open questions via live input.

In [ ]:
# ── STAGE 3b: OPEN QUESTIONS TO PIL ─────────────────────────────────────────

print("═" * 60)
print("STAGE 3b — OPEN QUESTIONS")
print("═" * 60)

open_questions = brief.get("open_questions", [])

if not open_questions:
    print("✓ No open questions — proceeding to ideation")
else:
    questions_task = (
        f"The creative brief is confirmed. Before ideation begins, surface these "
        f"open questions to the PIL. Present them conversationally — genuine curPrisimity, "
        f"not a form. Make clear that answers will shape the work, and that partial "
        f"answers or 'I don't know yet' are fine.\n\n"
        f"OPEN QUESTIONS:\n" + "\n".join(f"- {q}" for q in open_questions)
    )

    questions_response = call_role(
        role="director",
        user_message=questions_task,
        context=brief["challenge"],
        blackboard=blackboard,
        model=DIRECTOR_MODEL,
        max_tokens=300,
        brief_doc=read_brief_doc(session_id),
    )

    print("── DIRECTOR ASKS ───────────────────────────────────────────")
    print(questions_response)
    print()

    pil_answers = input("> ").strip()

    # Store and log
    brief["open_questions"].append(f"PIL answered: {pil_answers}")
    update_brief_doc(
        session_id,
        "DIRECTOR",
        "PIL OPEN QUESTION RESPONSES",
        f"**Director asked:**\n{questions_response}\n\n"
        f"**PIL responded:**\n{pil_answers}",
    )
    scribe_log(
        blackboard,
        "DIRECTOR",
        "open_questions_answered",
        f"Open questions answered. Context added to brief and Studio Brief Document.",
    )

    print()
    print("✓ Open questions answered — brief enriched")
    print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

---
## Cell 5 — Stage 4a: Creator

The creative brief goes directly to the Creator first — not the Researcher.
The Creator generates candidate directions informed by the brief.
Verbalized Sampling applied: probability weights push toward lateral thinking.
The Researcher has not yet acted — the Creative Team works first.

In [ ]:
# ── STAGE 4a: CREATOR — IDEATION PASS 1 ──────────────────────────────────────

print("═" * 60)
print("STAGE 4a — CREATOR: IDEATION (PASS 1)")
print("═" * 60)

creator_task = f"""You are working on this creative brief:

CHALLENGE: {brief['challenge']}
CONTEXT: {brief['context']}
DESIRED RESULT: {brief['desired_result']}
CONSTRAINTS: {", ".join(brief['constraints'])}

Generate 3 distinct conceptual directions using Verbalized Sampling.

━━━ VERBALIZED SAMPLING — ASSIGN THE BAND FIRST, THEN WRITE THE DIRECTION ━━━

One direction per band. Assign the band before writing. No two directions share a band.
The band determines the kind of thinking required — not a label applied afterward.

HIGH BAND (0.55–0.80)
The intelligent expected answer. A skilled practitioner working this brief carefully
would likely arrive here. Credible, grounded, implementable. Not obvious — but not
surprising. This is the direction that earns trust.

MID BAND (0.25–0.50)
A genuine reframe. The same problem understood differently. Requires stepping outside
the category's conventional assumptions. Not a variation on the high band direction —
a different way of understanding what the brief is actually asking for.

LOW BAND (0.08–0.22)
The direction that arrives from outside the expected territory entirely. Precise and
specific — lateral thinking is not loose thinking. This direction should make the room
go quiet. Uncomfortable and exact. A conventional approach would rarely reach here.

Assign your score within the band honestly. If it feels like 0.14, say 0.14.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

For each direction:
IDEA_ID: [A1, A2, A3]
BAND: [HIGH / MID / LOW]
PROBABILITY: [score within band]
TITLE: [short evocative name]
CONCEPT: [2-3 sentences — what is this direction?]
RATIONALE: [why this is worth exploring]
EMOTIONAL TERRITORY: [what feeling does this occupy?]
RESEARCH_REQUEST: [optional — one specific research question if external knowledge
would materially strengthen or ground this direction. Omit entirely if not needed.]

Do not write any preamble or analysis before your directions. Start directly with IDEA_ID: A1.

Be genuinely distinct — not three variations of the same idea."""

creator_response = call_role(
    role="creator",
    user_message=creator_task,
    context=brief["challenge"],
    blackboard=blackboard,
    model=SESSION_MODEL,
    max_tokens=1800,
    trait_profile=trait_profiles["creator"],
    brief_doc=read_brief_doc(session_id),
)

# Store as cycle 1
cycle_1 = {
    "cycle_number": 1,
    "creator_proposals": [{"raw_response": creator_response}],
    "critic_feedback": [],
    "synthesized_directions": [],
}
blackboard["ideation_cycles"].append(cycle_1)
update_brief_doc(session_id, "CREATOR", "DIRECTIONS — PASS 1", creator_response)
scribe_log(
    blackboard,
    "CREATOR",
    "ideation_complete",
    "Three directions generated using banded Verbalized Sampling. Cycle 1 logged.",
)
validate_stage(blackboard, "ideation")

print("── CREATOR DIRECTIONS ──────────────────────────────────────")
print(creator_response)
print("────────────────────────────────────────────────────────────")
print(f"\n✓ Creator pass 1 complete")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

---
## Cell 6 — Stage 4b: Critic

The Critic evaluates Creator directions and proposes a synthesis.
Reads the full Studio Brief Document including Creator output.
Constructive pressure — not rejection.

In [ ]:
# ── STAGE 4b: CRITIC — EVALUATION PASS 1 ────────────────────────────────────

print("═" * 60)
print("STAGE 4b — CRITIC: EVALUATION (PASS 1)")
print("═" * 60)

creator_output = blackboard["ideation_cycles"][-1]["creator_proposals"][0][
    "raw_response"
]

critic_task = f"""The Creator has proposed three directions for the brief:

BRIEF CHALLENGE: {brief['challenge']}
DESIRED RESULT: {brief['desired_result']}

CREATOR\'S DIRECTIONS:
{creator_output}

Evaluate each direction using this tight format:

IDEA_ID: [matching the Creator\'s ID]
WHAT HOLDS: [1-2 sentences — what genuinely works and why]
WHAT DOESN\'T: [1-2 sentences — the specific weakness or risk]
REFINEMENT PATH: [one concrete action that would strengthen this direction]
RESEARCH_REQUEST: [optional — one specific research question if external knowledge
would materially help evaluate or strengthen this direction. Omit if not needed.]

After all three evaluations, note any cross-direction patterns if present:
PATTERN: [optional — if multiple directions share a common strength or weakness worth naming]

Be rigorous and efficient. Identify the sharpest refinement path for each direction."""

critic_response = call_role(
    role="critic",
    user_message=critic_task,
    context=brief["challenge"],
    blackboard=blackboard,
    model=SESSION_MODEL,
    max_tokens=2400,
    trait_profile=trait_profiles["critic"],
    brief_doc=read_brief_doc(session_id),
)

blackboard["ideation_cycles"][-1]["critic_feedback"].append(
    {"raw_response": critic_response}
)
update_brief_doc(session_id, "CRITIC", "EVALUATION — PASS 1", critic_response)
scribe_log(
    blackboard,
    "CRITIC",
    "evaluation_complete",
    "Pass 1 evaluation complete. Refinement paths identified.",
)
validate_stage(blackboard, "critic")

# Parse RESEARCH_REQUESTs from Creator and Critic for Researcher routing
creator_requests = re.findall(r"RESEARCH_REQUEST:\s*(.+?)(?=\n|$)", creator_output)
critic_requests = re.findall(r"RESEARCH_REQUEST:\s*(.+?)(?=\n|$)", critic_response)
research_requests = [
    f"[Creator]: {r.strip()}" for r in creator_requests if r.strip()
] + [f"[Critic]:  {r.strip()}" for r in critic_requests if r.strip()]

print("── CRITIC EVALUATION ───────────────────────────────────────")
print(critic_response)
print("────────────────────────────────────────────────────────────")
print(f"\n✓ Critic pass 1 complete")
if research_requests:
    print(f"  RESEARCH_REQUESTs found: {len(research_requests)}")
    for r in research_requests:
        print(f"    {r}")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

---
## Cell 7 — Stage 4c: Researcher

The Researcher acts AFTER the Creative Team — not before.
It reads the full Studio Brief Document including the ideation cycle
and provides contextual enrichment informed by what the team actually produced.
This is also when the bounded autonomous contribution trigger is evaluated:
the Researcher now has sufficient thinking on the Blackboard to identify genuine gaps.

In [ ]:
# ── STAGE 4c: RESEARCHER ─────────────────────────────────────────────────────

print("═" * 60)
print("STAGE 4c — RESEARCHER")
print("═" * 60)

brief = blackboard["creative_brief"]
creator_output = blackboard["ideation_cycles"][-1]["creator_proposals"][0][
    "raw_response"
]
critic_output = blackboard["ideation_cycles"][-1]["critic_feedback"][0]["raw_response"]

# Build request section if RESEARCH_REQUESTs were found
requests_section = ""
if research_requests:
    requests_section = (
        "\n\nSPECIFIC RESEARCH REQUESTS FROM THE CREATIVE TEAM:\n"
        + "\n".join(f"- {r}" for r in research_requests)
        + "\n\nAddress each of these specific requests directly before making any "
        "autonomous contribution."
    )

research_task = (
    f"You are supporting a creative studio session.\n\n"
    f"THE BRIEF:\n"
    f"Challenge: {brief['challenge']}\n"
    f"Context: {brief['context']}\n"
    f"Desired result: {brief['desired_result']}\n"
    f"{requests_section}\n\n"
    f"THE CREATIVE TEAM HAS PRODUCED:\n\n"
    f"CREATOR'S DIRECTIONS:\n{creator_output}\n\n"
    f"CRITIC'S EVALUATION:\n{critic_output}\n\n"
    f"Your task: source and cite specific external precedents, examples, or factual "
    f"findings that ground or expand what the Creative Team has produced.\n\n"
    f"For each finding:\n"
    f"TOPIC: [what area, domain, or precedent]\n"
    f"SOURCE: [name the specific source, study, institution, or example — be precise.\n"
    f"         Not 'research suggests' but the actual named source.]\n"
    f"FINDING: [what it shows]\n"
    f"RELEVANCE: [why this matters specifically for these directions]\n\n"
    f"Address specific requests first. Then make one autonomous contribution if you "
    f"have genuinely relevant external knowledge not already covered.\n\n"
    f"If the Creative Team's work is already well-grounded and no external knowledge "
    f"would add to it, say only:\n"
    f"RESEARCHER: No external contribution warranted for this cycle.\n\n"
    f"You are a sourcing agent. Do not evaluate the Creative Team's directions."
)

research_response = call_role(
    role="researcher",
    user_message=research_task,
    context=brief["challenge"],
    blackboard=blackboard,
    model=SESSION_MODEL,
    max_tokens=2000,
    trait_profile=trait_profiles["researcher"],
    brief_doc=read_brief_doc(session_id),
)

research_entry = {
    "research_id": f"R{len(blackboard['research_trace']) + 1:03d}",
    "initiated_by": "creative_team",
    "query": "Targeted requests + autonomous contribution post-ideation",
    "sources": ["researcher_knowledge"],
    "summary": research_response,
    "influence_on_ideation": "Delivered to Creative Team for Pass 2 refinement",
}
blackboard["research_trace"].append(research_entry)
brief["research_insights"].append(research_response)
update_brief_doc(session_id, "RESEARCHER", "RESEARCH FINDINGS", research_response)
scribe_log(
    blackboard,
    "RESEARCHER",
    "research_delivered",
    f"Research delivered. Entry R{len(blackboard['research_trace']):03d} logged.",
)

print("── RESEARCHER FINDINGS ─────────────────────────────────────")
print(research_response)
print("────────────────────────────────────────────────────────────")
print(f"\n✓ Research logged")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

---
## Cell 7b — Stage 4d: Creator Refinement (Pass 2)

Creator reads the Critic's evaluation and the Researcher's findings.
Refines each of the three directions — does not replace them.
Incorporates research where relevant, addresses Critic's refinement paths.

In [ ]:
# ── STAGE 4d: CREATOR — REFINEMENT PASS 2 ─────────────────────────────

print("═" * 60)
print("STAGE 4d — CREATOR: REFINEMENT (PASS 2)")
print("═" * 60)

creator_refinement_task = (
    f"You are the Creator. The Critic has evaluated your directions and the "
    f"Researcher has delivered findings. Now refine your work.\n\n"
    f"YOUR ORIGINAL DIRECTIONS:\n{creator_output}\n\n"
    f"CRITIC'S EVALUATION:\n{critic_output}\n\n"
    f"RESEARCHER'S FINDINGS:\n{research_response}\n\n"
    f"Refine each of your three directions. Do not replace them — refine them.\n"
    f"For each direction:\n"
    f"- Incorporate what the research adds, where it is genuinely relevant\n"
    f"- Address the Critic's refinement path\n"
    f"- Sharpen the concept based on what you now know\n\n"
    f"Use the same format as before:\n"
    f"IDEA_ID: [A1, A2, A3 — same IDs]\n"
    f"BAND: [same band — the territory doesn't change]\n"
    f"PROBABILITY: [may adjust within band if refinement shifted the direction]\n"
    f"TITLE: [may refine]\n"
    f"CONCEPT: [refined — 2-3 sentences]\n"
    f"RATIONALE: [updated to incorporate research and critique]\n"
    f"EMOTIONAL TERRITORY: [same or refined]\n\n"
    f"If a direction is already strong and the research adds nothing to it, "
    f"say so in one sentence and keep it as written. Refinement is not mandatory "
    f"where the direction already holds."
)

creator_refined_response = call_role(
    role="creator",
    user_message=creator_refinement_task,
    context=brief["challenge"],
    blackboard=blackboard,
    model=SESSION_MODEL,
    max_tokens=2400,  # raised from 1200 — 3-direction Pass 2 needs room
    trait_profile=trait_profiles["creator"],
    brief_doc=read_brief_doc(session_id),
)

# Store as cycle 2 (research-integrated refinement)
cycle_2 = {
    "cycle_number": 2,
    "creator_proposals": [{"raw_response": creator_refined_response}],
    "critic_feedback": [],
    "synthesized_directions": [],
}
blackboard["ideation_cycles"].append(cycle_2)
update_brief_doc(
    session_id, "CREATOR", "DIRECTIONS — PASS 2 (REFINED)", creator_refined_response
)
scribe_log(
    blackboard,
    "CREATOR",
    "refinement_complete",
    "Pass 2 refinement complete. Research and critique incorporated.",
)
validate_stage(blackboard, "ideation")

# Update creator_output to point to refined version for downstream cells
creator_output = creator_refined_response

print("── CREATOR REFINED DIRECTIONS ──────────────────────────────────")
print(creator_refined_response)
print("────────────────────────────────────────────────────────────────")
print(f"\n✓ Creator pass 2 complete")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

---
## Cell 7c — Stage 4e: Critic Final Evaluation + Synthesis (Pass 2)

Critic evaluates the refined directions and produces one synthesis direction.
This is the Creative Team's final submission to the Director.
The synthesis direction may become a fourth candidate in the Director's review.

In [ ]:
# ── STAGE 4e: CRITIC — FINAL EVALUATION + SYNTHESIS (PASS 2) ────────────────

print("═" * 60)
print("STAGE 4e — CRITIC: FINAL EVALUATION + SYNTHESIS (PASS 2)")
print("═" * 60)

critic_synthesis_task = (
    f"You are the Critic. The Creator has refined their three directions based on "
    f"your evaluation and the Researcher's findings.\n\n"
    f"BRIEF CHALLENGE: {brief['challenge']}\n"
    f"DESIRED RESULT: {brief['desired_result']}\n\n"
    f"REFINED DIRECTIONS:\n{creator_output}\n\n"
    f"This is the Creative Team's final submission before Director review.\n\n"
    f"For each refined direction:\n"
    f"IDEA_ID: [matching ID]\n"
    f"ASSESSMENT: [1-2 sentences — does the refinement hold? Is it stronger than before?]\n"
    f"REMAINING CONCERN: [one sentence if any weakness persists — or 'None']\n\n"
    f"Then produce ONE synthesis direction — something that combines the strongest "
    f"elements across the three refined directions into something none of them "
    f"reached alone:\n\n"
    f"SYNTHESIS_ID: S1\n"
    f"TITLE: [name]\n"
    f"CONCEPT: [2-3 sentences]\n"
    f"ORIGIN_IDEAS: [which direction IDs it draws from]\n"
    f"WHY THIS: [one sentence — what does this offer that the individual directions don't]\n\n"
    f"Present all four directions (A1, A2, A3, S1) with their trade-offs stated "
    f"factually -- do not rank or recommend. Close with:\n"
    f"The Creative Team submits these four directions to the Director for review."
)

critic_synthesis_response = call_role(
    role="critic",
    user_message=critic_synthesis_task,
    context=brief["challenge"],
    blackboard=blackboard,
    model=SESSION_MODEL,
    max_tokens=2400,
    trait_profile=trait_profiles["critic"],
    brief_doc=read_brief_doc(session_id),
)

blackboard["ideation_cycles"][-1]["critic_feedback"].append(
    {"raw_response": critic_synthesis_response}
)
update_brief_doc(
    session_id,
    "CRITIC",
    "FINAL EVALUATION + SYNTHESIS (PASS 2)",
    critic_synthesis_response,
)
scribe_log(
    blackboard,
    "CRITIC",
    "synthesis_complete",
    "Pass 2 final evaluation and synthesis complete. Ready for Director review.",
)
validate_stage(blackboard, "critic")

# Update critic_output to refined version for downstream cells
critic_output = critic_synthesis_response

print("── CRITIC FINAL EVALUATION + SYNTHESIS ─────────────────────")
print(critic_synthesis_response)
print("────────────────────────────────────────────────────────────")
print(f"\n✓ Critic pass 2 complete — Creative Team submission ready")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

---
## Cell 8 — Stage 5 & 6: Candidate Set + Director Review

Director assembles candidate directions from the ideation cycle,
evaluates them against the brief, and decides whether to present
or send back for another round.

In [ ]:
# ── STAGE 5: CANDIDATE DIRECTIONS ────────────────────────────────────────────
# ── STAGE 6: DIRECTOR REVIEW ─────────────────────────────────────────────────

print("═" * 60)
print("STAGE 5 — CANDIDATE DIRECTIONS")
print("STAGE 6 — DIRECTOR REVIEW")
print("═" * 60)

# Read from the latest ideation cycle (cycle 2 — refined)
creator_output = blackboard["ideation_cycles"][-1]["creator_proposals"][-1][
    "raw_response"
]
critic_output = blackboard["ideation_cycles"][-1]["critic_feedback"][-1]["raw_response"]

review_task = f"""You are reviewing the Creative Team\'s final submission before presenting to the PIL.
The team has completed two passes: initial ideation, research integration, and refinement.

BRIEF:
Challenge: {brief['challenge']}
Desired result: {brief['desired_result']}
Constraints: {", ".join(brief['constraints'])}

REFINED CREATOR DIRECTIONS:
{creator_output[:900]}

CRITIC FINAL EVALUATION + SYNTHESIS:
{critic_output[:900]}

Your task:
1. Select 3 directions to present — from the refined set plus the Critic\'s synthesis (S1).
   Include S1 if it is stronger than any of the three refined directions.
2. Evaluate the candidate set.
3. Decide: PRESENT or ITERATE.

Return ONLY a JSON object — no preamble, no markdown fences:

{{
  "candidates": [
    {{"id": "C1", "title": "exact direction name", "type": "evolutionary|evolutionary_variant|conceptual_leap|sacrificial_probe", "source_idea_id": "A1"}},
    {{"id": "C2", "title": "...", "type": "...", "source_idea_id": "A2"}},
    {{"id": "C3", "title": "...", "type": "...", "source_idea_id": "S1"}}
  ],
  "alignment": "one sentence",
  "distinctiveness": "one sentence",
  "balance": "one sentence",
  "clarity": "one sentence",
  "decision": "PRESENT",
  "reason": "one sentence"
}}

Set decision to "ITERATE" with reason if the team\'s work does not meet standard."""

review_response = call_role(
    role="director",
    user_message=review_task,
    context=brief["challenge"],
    blackboard=blackboard,
    model=DIRECTOR_MODEL,
    max_tokens=800,
    brief_doc=read_brief_doc(session_id),
)

# ── Parse Director review — JSON with validation gate ─────────────────────────
review = blackboard["director_review"]

try:
    clean = review_response.strip()
    if clean.startswith("```"):
        clean = clean.split("```")[1]
        if clean.startswith("json"):
            clean = clean[4:]
    parsed = json.loads(clean.strip())

    review["alignment_with_brief"] = parsed.get("alignment", "")
    review["distinctiveness_assessment"] = parsed.get("distinctiveness", "")
    review["balance_assessment"] = parsed.get("balance", "")
    review["clarity_assessment"] = parsed.get("clarity", "")
    decision = parsed.get("decision", "PRESENT")
    review["iteration_required"] = "ITERATE" in decision.upper()

    candidates = parsed.get("candidates", [])
    if not candidates:
        raise ValueError("Director review returned no candidates")

    for c in candidates:
        blackboard["candidate_set"].append(
            {
                "direction_id": c.get(
                    "id", f"D{len(blackboard['candidate_set'])+1:03d}"
                ),
                "internal_type": c.get("type", "evolutionary").lower(),
                "description": c.get("title", ""),
                "reasoning_summary": "",
                "research_influences": [],
                "critic_assessment": {
                    "strengths": [],
                    "concerns": [],
                    "refinement_notes": [],
                },
            }
        )

    print("✓ Director review parsed successfully")
    print(f"  Decision   : {decision}")
    print(f"  Candidates : {len(candidates)}")

except (json.JSONDecodeError, ValueError) as e:
    print(f"✗ Director review parsing failed: {e}")
    print(review_response)
    raise RuntimeError("Director review parsing failed — candidate set empty.")

update_brief_doc(session_id, "DIRECTOR", "DIRECTOR REVIEW", review_response)
scribe_log(
    blackboard,
    "DIRECTOR",
    "review_complete",
    f"Director review complete. Decision: {'ITERATE' if review['iteration_required'] else 'PRESENT'}. "
    f"{len(blackboard['candidate_set'])} candidates selected.",
)

if not review["iteration_required"]:
    validate_stage(blackboard, "candidate_set")

print("── DIRECTOR REVIEW ─────────────────────────────────────────")
print(review_response)
print("────────────────────────────────────────────────────────────")
print(f"\n✓ Director review complete")
print(f"  Candidates selected : {len(blackboard['candidate_set'])}")
print(f"  Iteration required  : {review['iteration_required']}")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

---
## Cell 8a — Stage 6a: Iteration Loop

Fires only when the Director signals ITERATE.
Re-runs the full Creative Team: Creator → Critic → Researcher.
Then re-runs Director review to produce a new candidate set.
Cell 9 (Presentation) proceeds normally from the updated candidate_set.

If ITERATE is not required, this cell prints a confirmation and passes through.

In [ ]:
# ── STAGE 6a: ITERATION LOOP ─────────────────────────────────────────────────

print("═" * 60)
print("STAGE 6a — ITERATION LOOP")
print("═" * 60)

if not review["iteration_required"]:
    print("✓ Director decision: PRESENT — no iteration needed")
    print(f"✓ Candidates in set: {len(blackboard['candidate_set'])}")

else:
    print("⚠ Director decision: ITERATE — re-running full Creative Team")
    print()

    # Clear existing ideation cycle and candidate set for a clean cycle 2
    blackboard["ideation_cycles"] = []
    blackboard["candidate_set"] = []

    # ── CREATOR — CYCLE 2 ─────────────────────────────────────────────────────
    print("── CREATOR: CYCLE 2 ────────────────────────────────────────")

    iteration_context = (
        f"The Director reviewed the first ideation cycle and requested a full re-run.\n\n"
        f"DIRECTOR'S FEEDBACK:\n{review_response}\n\n"
        f"BRIEF:\n"
        f"Challenge: {brief['challenge']}\n"
        f"Context: {brief['context']}\n"
        f"Desired result: {brief['desired_result']}\n"
        f"Constraints: {', '.join(brief['constraints'])}\n\n"
        f"Generate 4 new distinct directions that address the Director's specific concerns above.\n"
        f"Use Verbalized Sampling: include probability scores. Include at least one below 0.15.\n\n"
        f"For each direction:\n"
        f"IDEA_ID: [A1, A2, A3, A4]\n"
        f"TITLE: [short evocative name]\n"
        f"PROBABILITY: [0.0-1.0]\n"
        f"CONCEPT: [2-3 sentences]\n"
        f"RATIONALE: [why this is worth exploring]\n"
        f"EMOTIONAL TERRITORY: [what feeling does this occupy?]"
    )

    creator_response_iter = call_role(
        role="creator",
        user_message=iteration_context,
        context=brief["challenge"],
        blackboard=blackboard,
        model=SESSION_MODEL,
        max_tokens=1200,
        trait_profile=trait_profiles["creator"],
        brief_doc=read_brief_doc(session_id),
    )

    cycle_2 = {
        "cycle_number": 2,
        "creator_proposals": [{"raw_response": creator_response_iter}],
        "critic_feedback": [],
        "synthesized_directions": [],
    }
    blackboard["ideation_cycles"].append(cycle_2)
    update_brief_doc(
        session_id, "CREATOR", "DIRECTIONS — CYCLE 2", creator_response_iter
    )
    scribe_log(
        blackboard,
        "CREATOR",
        "iteration_ideation_complete",
        "Cycle 2 ideation complete following Director iteration request.",
    )
    validate_stage(blackboard, "ideation")

    print(creator_response_iter)
    print()

    # ── CRITIC — CYCLE 2 ──────────────────────────────────────────────────────
    print("── CRITIC: CYCLE 2 ─────────────────────────────────────────")

    critic_task_iter = (
        f"The Creator has proposed a new set of directions following Director feedback.\n\n"
        f"BRIEF CHALLENGE: {brief['challenge']}\n"
        f"DESIRED RESULT: {brief['desired_result']}\n\n"
        f"CREATOR'S NEW DIRECTIONS:\n{creator_response_iter}\n\n"
        f"Evaluate each direction. For each:\n"
        f"IDEA_ID: [matching the Creator's ID]\n"
        f"STRENGTHS: [what genuinely works]\n"
        f"WEAKNESSES: [what is underdeveloped or misaligned]\n"
        f"REFINEMENT: [one concrete suggestion]\n\n"
        f"Then propose ONE synthesis direction:\n"
        f"SYNTHESIS_ID: S1\n"
        f"TITLE: [name]\n"
        f"CONCEPT: [2-3 sentences]\n"
        f"ORIGIN_IDEAS: [which Creator ideas it draws from]\n\n"
        f"Be analytically rigorous."
    )

    critic_response_iter = call_role(
        role="critic",
        user_message=critic_task_iter,
        context=brief["challenge"],
        blackboard=blackboard,
        model=SESSION_MODEL,
        max_tokens=1800,
        trait_profile=trait_profiles["critic"],
        brief_doc=read_brief_doc(session_id),
    )

    blackboard["ideation_cycles"][-1]["critic_feedback"].append(
        {"raw_response": critic_response_iter}
    )
    update_brief_doc(session_id, "CRITIC", "CRITIQUE — CYCLE 2", critic_response_iter)
    scribe_log(
        blackboard,
        "CRITIC",
        "iteration_evaluation_complete",
        "Cycle 2 Critic evaluation complete.",
    )
    validate_stage(blackboard, "critic")

    print(critic_response_iter)
    print()

    # ── RESEARCHER — CYCLE 2 ──────────────────────────────────────────────────
    print("── RESEARCHER: CYCLE 2 ─────────────────────────────────────")

    research_task_iter = (
        f"You are supporting a creative studio session. The Creative Team has completed "
        f"a second ideation cycle.\n\n"
        f"THE BRIEF:\n"
        f"Challenge: {brief['challenge']}\n"
        f"Context: {brief['context']}\n"
        f"Desired result: {brief['desired_result']}\n\n"
        f"CREATOR'S DIRECTIONS (CYCLE 2):\n{creator_response_iter}\n\n"
        f"CRITIC'S EVALUATION (CYCLE 2):\n{critic_response_iter}\n\n"
        f"Your task: source and cite 2-3 external precedents, examples, or factual findings\n"
        f"from outside this problem's domain that could ground or expand what the Creative\n"
        f"Team has produced. Respond to the actual directions above.\n\n"
        f"For each finding:\n"
        f"TOPIC: [what area, domain, or precedent]\n"
        f"SOURCE: [name the specific source, study, institution, or example — be precise]\n"
        f"FINDING: [what it shows]\n"
        f"RELEVANCE: [why this matters for these specific directions]\n\n"
        f"If you have no external knowledge that would genuinely add here, say only:\n"
        f"RESEARCHER: No external contribution warranted for this cycle.\n\n"
        f"You are a sourcing agent, not an analyst. Do not evaluate the Creative Team's work."
    )

    research_response_iter = call_role(
        role="researcher",
        user_message=research_task_iter,
        context=brief["challenge"],
        blackboard=blackboard,
        model=SESSION_MODEL,
        max_tokens=700,
        trait_profile=trait_profiles["researcher"],
        brief_doc=read_brief_doc(session_id),
    )

    research_entry_iter = {
        "research_id": f"R{len(blackboard['research_trace']) + 1:03d}",
        "initiated_by": "director_iteration",
        "query": "Post-ideation cycle 2 enrichment",
        "sources": ["researcher_knowledge"],
        "summary": research_response_iter,
        "influence_on_ideation": "Cycle 2 post-iteration research",
    }
    blackboard["research_trace"].append(research_entry_iter)
    brief["research_insights"].append(research_response_iter)
    update_brief_doc(
        session_id, "RESEARCHER", "RESEARCH — CYCLE 2", research_response_iter
    )
    scribe_log(
        blackboard,
        "RESEARCHER",
        "iteration_research_delivered",
        f"Cycle 2 research. Entry R{len(blackboard['research_trace']):03d} logged.",
    )

    print(research_response_iter)
    print()

    # ── DIRECTOR REVIEW — CYCLE 2 ─────────────────────────────────────────────
    print("── DIRECTOR REVIEW: CYCLE 2 ────────────────────────────────")

    review_task_iter = (
        f"You are reviewing the studio's second ideation cycle before presenting to the PIL.\n\n"
        f"BRIEF:\n"
        f"Challenge: {brief['challenge']}\n"
        f"Desired result: {brief['desired_result']}\n"
        f"Constraints: {', '.join(brief['constraints'])}\n\n"
        f"CREATOR DIRECTIONS (CYCLE 2):\n{creator_response_iter[:800]}\n\n"
        f"CRITIC EVALUATION (CYCLE 2):\n{critic_response_iter[:800]}\n\n"
        f"Select 3 directions to present. Return ONLY a JSON object — "
        f"no preamble, no markdown fences:\n\n"
        f"{{\n"
        f'  "candidates": [\n'
        f'    {{"id": "C1", "title": "...", "type": "evolutionary|evolutionary_variant|conceptual_leap|sacrificial_probe", "source_idea_id": "A1"}},\n'
        f'    {{"id": "C2", "title": "...", "type": "...", "source_idea_id": "A2"}},\n'
        f'    {{"id": "C3", "title": "...", "type": "...", "source_idea_id": "S1"}}\n'
        f"  ],\n"
        f'  "alignment": "...",\n'
        f'  "distinctiveness": "...",\n'
        f'  "balance": "...",\n'
        f'  "clarity": "...",\n'
        f'  "decision": "PRESENT",\n'
        f'  "reason": "..."\n'
        f"}}"
    )

    review_response_iter = call_role(
        role="director",
        user_message=review_task_iter,
        context=brief["challenge"],
        blackboard=blackboard,
        model=DIRECTOR_MODEL,
        max_tokens=800,
        brief_doc=read_brief_doc(session_id),
    )

    # Parse cycle 2 Director review
    try:
        clean_iter = review_response_iter.strip()
        if clean_iter.startswith("```"):
            clean_iter = clean_iter.split("```")[1]
            if clean_iter.startswith("json"):
                clean_iter = clean_iter[4:]
        parsed_iter = json.loads(clean_iter.strip())

        review["alignment_with_brief"] = parsed_iter.get("alignment", "")
        review["distinctiveness_assessment"] = parsed_iter.get("distinctiveness", "")
        review["balance_assessment"] = parsed_iter.get("balance", "")
        review["clarity_assessment"] = parsed_iter.get("clarity", "")
        review["iteration_required"] = False  # reset after cycle 2 review

        for c in parsed_iter.get("candidates", []):
            blackboard["candidate_set"].append(
                {
                    "direction_id": c.get(
                        "id", f"D{len(blackboard['candidate_set'])+1:03d}"
                    ),
                    "internal_type": c.get("type", "evolutionary").lower(),
                    "description": c.get("title", ""),
                    "reasoning_summary": "",
                    "research_influences": [],
                    "critic_assessment": {
                        "strengths": [],
                        "concerns": [],
                        "refinement_notes": [],
                    },
                }
            )

        print("✓ Cycle 2 Director review parsed successfully")

    except (json.JSONDecodeError, ValueError) as e:
        print(f"✗ Cycle 2 Director review parsing failed: {e}")
        print(review_response_iter)
        raise RuntimeError(
            "Iteration Director review failed — candidate set still empty.\n"
            "Check the Director prompt or model output above."
        )

    update_brief_doc(
        session_id, "DIRECTOR", "DIRECTOR REVIEW — CYCLE 2", review_response_iter
    )
    scribe_log(
        blackboard,
        "DIRECTOR",
        "iteration_review_complete",
        f"Cycle 2 Director review complete. {len(blackboard['candidate_set'])} candidates selected.",
    )
    validate_stage(blackboard, "candidate_set")

    # Update creator_output and critic_output so downstream cells (9 and 11)
    # read from cycle 2, not cycle 1.
    creator_output = creator_response_iter
    critic_output = critic_response_iter

    print()
    print(f"✓ Iteration complete")
    print(f"✓ Candidates in set : {len(blackboard['candidate_set'])}")
    print(f"✓ Reasoning trace   : {len(blackboard['reasoning_trace'])} entries")

---
## Cell 9 — Stage 7: Presentation

Director presents the candidate directions to the PIL.
Each direction includes reasoning. Tone is clear, professional, non-salesy.

In [ ]:
# ── STAGE 7: PRESENTATION ────────────────────────────────────────────────────

print("═" * 60)
print("STAGE 7 — PRESENTATION")
print("═" * 60)

candidate_summary = "\n".join(
    [
        f"- {c['description']} (type: {c['internal_type']})"
        for c in blackboard["candidate_set"]
    ]
)

presentation_task = f"""Present these three directions to the person in the loop.

BRIEF CHALLENGE: {brief['challenge']}

CANDIDATE DIRECTIONS:
{candidate_summary}

FULL STUDIO OUTPUT TO DRAW FROM:
Creator: {creator_output[:600]}
Critic: {critic_output[:600]}

Write a clear, engaging presentation. For each direction:
- Give it a memorable name
- Describe it in 2-3 sentences the PIL can actually picture
- Explain briefly why it deserves consideration

Sequence matters: lead with the strongest, end with the most unexpected.
Tone: warm, direct, confident. No hype. No 'exciting' or 'powerful'.
Close with a single question that invites reaction."""

presentation_response = call_role(
    role="director",
    user_message=presentation_task,
    context=brief["challenge"],
    blackboard=blackboard,
    model=DIRECTOR_MODEL,
    max_tokens=1000,
    brief_doc=read_brief_doc(session_id),
)

# Store presentation
blackboard["presentation"]["ordered_directions"] = [
    c["direction_id"] for c in blackboard["candidate_set"]
]
blackboard["presentation"]["director_commentary"].append(presentation_response)

scribe_log(
    blackboard,
    "DIRECTOR",
    "presentation_delivered",
    f"Presentation delivered to PIL. {len(blackboard['candidate_set'])} directions presented.",
)

print("── DIRECTOR PRESENTS TO PIL ────────────────────────────────")
print(presentation_response)
print("────────────────────────────────────────────────────────────")
print(f"\n✓ Presentation complete")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

---
## Cell 10 — Stage 8: User Reaction

PIL responds to the candidate directions.
Director extracts creative signals from the reaction.
These signals determine whether to proceed to synthesis or trigger a second loop.

In [ ]:
# ── STAGE 8: USER REACTION ─────────────────────────────────────────────

print("═" * 60)
print("STAGE 8 — USER REACTION")
print("═" * 60)

pil_reaction = input("Your reaction: ").strip()
print()

# Director extracts signals — JSON format prevents markdown parsing failures
signal_task = f"""The PIL responded to the presentation:
"{pil_reaction}"

Extract the creative signals from this reaction.

Return ONLY a JSON object — no preamble, no markdown fences:

{{
  "selected_direction": "<which direction resonated, e.g. C1, C2, C3, or none>",
  "signal_1": "<key preference or instinct revealed>",
  "signal_2": "<what they want more or less of>",
  "signal_3": "<any new insight or direction revealed, or empty string>",
  "second_loop": <true if the reaction reveals new information that would meaningfully change the creative directions, false if the reaction confirms an existing direction>,
  "second_loop_reason": "<why second loop is or isn't needed — one sentence>"
}}

second_loop should be true only if the PIL's reaction reveals something genuinely new
that the Creative Team did not have when they built the directions — a reframe, a
constraint, a signal that substantially changes what the synthesis should be.
It should NOT trigger just because the PIL wants to push a direction further."""

signal_response = call_role(
    role="director",
    user_message=signal_task,
    context=brief["challenge"],
    blackboard=blackboard,
    model=DIRECTOR_MODEL,
    max_tokens=800,  # raised from 400 — JSON response needs room
    brief_doc=read_brief_doc(session_id),
)

# Parse JSON response — with fallback if Director returns narrative instead
try:
    clean = signal_response.strip()
    if clean.startswith("```"):
        clean = clean.split("```")[1]
        if clean.startswith("json"):
            clean = clean[4:]
    signal_data = json.loads(clean.strip())
    second_loop_needed = bool(signal_data.get("second_loop", False))
    second_loop_reason = signal_data.get("second_loop_reason", "")
    signals_list = [
        signal_data.get("signal_1", ""),
        signal_data.get("signal_2", ""),
        signal_data.get("signal_3", ""),
    ]
    signals_list = [s for s in signals_list if s.strip()]
    selected_dir = signal_data.get("selected_direction", "")
except (json.JSONDecodeError, ValueError, AttributeError):
    # Fallback: Director returned narrative — use as-is, no second loop
    second_loop_needed = False
    second_loop_reason = "JSON parse failed — narrative response stored"
    signals_list = []
    selected_dir = ""
    print("  ⚠ Signal extraction returned non-JSON — stored as narrative")

# Store user response
user_resp = blackboard["user_response"]
user_resp["feedback_summary"] = pil_reaction
user_resp["signals_extracted"].append(signal_response)
if selected_dir:
    user_resp["selected_direction"] = selected_dir

# Trigger second loop if needed
blackboard["second_exploration"]["triggered"] = second_loop_needed
if second_loop_needed:
    blackboard["second_exploration"]["reason"] = second_loop_reason

# Update Studio Brief Document
update_brief_doc(
    session_id,
    "DIRECTOR",
    "PIL SIGNALS",
    f"**PIL reaction:** {pil_reaction}\n\n"
    f"**Selected:** {selected_dir}\n\n"
    f"**Signals extracted:**\n{signal_response}\n\n"
    f"**Second loop:** {second_loop_needed}"
    + (f" — {second_loop_reason}" if second_loop_reason else ""),
)

scribe_log(
    blackboard,
    "DIRECTOR",
    "signals_extracted",
    f"PIL reaction processed. Second loop: {second_loop_needed}. "
    f"Key signal: '{pil_reaction[:60]}...'",
)

print("── CREATIVE SIGNALS EXTRACTED ──────────────────────────────────")
print(signal_response)
print("────────────────────────────────────────────────────────────────")
print(f"\n✓ Signals extracted")
print(f"  Selected direction  : {selected_dir or 'not parsed'}")
print(f"  Second loop         : {second_loop_needed}")
if second_loop_needed:
    print(f"  Reason              : {second_loop_reason}")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

---
## Cell 10b — Stage 8b: Depth Extraction

Director uses one extraction technique from the toolkit to probe deeper
before committing to final synthesis.
Director asks one focused question, PIL responds via live input.

In [ ]:
# ── STAGE 8b: DEPTH EXTRACTION ───────────────────────────────────────────────

print("═" * 60)
print("STAGE 8b — DEPTH EXTRACTION")
print("═" * 60)

depth_task = (
    f"The PIL has reacted to the presentation. Before producing the final synthesis,\n"
    f"probe one level deeper using a single extraction technique from your toolkit.\n\n"
    f"PIL REACTION:\n{pil_reaction}\n\n"
    f"SIGNALS EXTRACTED SO FAR:\n{signal_response}\n\n"
    f"Choose ONE technique that would surface the most useful additional signal right now.\n"
    f"Ask a single focused question — not multiple questions.\n"
    f"Keep it conversational. Do not explain why you're asking. Just ask."
)

depth_question = call_role(
    role="director",
    user_message=depth_task,
    context=brief["challenge"],
    blackboard=blackboard,
    model=DIRECTOR_MODEL,
    max_tokens=150,
    brief_doc=read_brief_doc(session_id),
)

print("── DIRECTOR DEPTH QUESTION ─────────────────────────────────")
print(depth_question)
print()
depth_response = input("> ").strip()

# Store depth signal alongside Stage 8 signals
blackboard["user_response"]["signals_extracted"].append(
    f"DEPTH SIGNAL (Stage 8b): {depth_response}"
)
update_brief_doc(
    session_id,
    "DIRECTOR",
    "DEPTH SIGNAL",
    f"**Director question:** {depth_question}\n\n"
    f"**PIL response:** {depth_response}",
)
scribe_log(
    blackboard,
    "DIRECTOR",
    "depth_extraction_complete",
    f"Stage 8b depth extraction complete. Signal: '{depth_response[:80]}'",
)

print()
print("✓ Depth signal captured")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")


---
## Cell 10c — Stage 8c: Second Loop
 
Fires only when the Director's signal extraction identified new information
that warrants a fresh Creative Team run.
 
If second_exploration.triggered is True: re-runs Creator → Critic → Researcher
with the PIL's reaction signals as additional brief context.
Updated directions replace the original candidate set for synthesis.
 
If False: passes through immediately. No API calls made.


In [ ]:
# ── STAGE 8c: SECOND LOOP ──────────────────────────────────────────────

print("═" * 60)
print("STAGE 8c — SECOND LOOP")
print("═" * 60)

if not blackboard["second_exploration"]["triggered"]:
    print("✓ Second loop not required — proceeding to synthesis")
    print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

else:
    print("⚠ Second loop triggered — re-running Creative Team with PIL signals\n")
    print(f"  Reason: {blackboard['second_exploration']['reason']}\n")

    # -- Assemble updated brief incorporating PIL signals ----------------
    all_signals = "\n\n".join(blackboard["user_response"]["signals_extracted"])

    second_loop_brief = (
        f"ORIGINAL BRIEF:\n"
        f"Challenge: {brief['challenge']}\n"
        f"Context: {brief['context']}\n"
        f"Desired result: {brief['desired_result']}\n"
        f"Constraints: {'; '.join(brief.get('constraints', []))}\n\n"
        f"NEW SIGNAL FROM PIL REACTION:\n"
        f"{all_signals}\n\n"
        f"The PIL has reacted to the first set of directions. Their reaction "
        f"revealed new information that changes what the directions should be. "
        f"You are starting a fresh ideation cycle informed by this new signal. "
        f"Do not repeat the previous directions. Build something new."
    )

    # -- Update Studio Brief Document ------------------------------------
    update_brief_doc(
        session_id,
        "DIRECTOR",
        "SECOND LOOP — PIL SIGNAL UPDATE",
        f"Second loop triggered.\n\nReason: "
        f"{blackboard['second_exploration']['reason']}\n\n"
        f"New signal:\n{all_signals}",
    )

    # ── Creator Pass 1 (Second Loop) ────────────────────────────────────
    print("── CREATOR: SECOND LOOP ─────────────────────────────────────")

    creator_task_2 = (
        f"You are the Creator. A new creative brief has emerged from the "
        f"person in the loop's reaction to the first set of directions.\n\n"
        f"{second_loop_brief}\n\n"
        f"Generate three fresh directions using Verbalized Sampling across "
        f"probability bands (HIGH / MID / LOW).\n\n"
        f"For each direction:\n"
        f"IDEA_ID: [B1, B2, B3]\n"
        f"BAND: [HIGH / MID / LOW]\n"
        f"PROBABILITY: [0.0–1.0]\n"
        f"TITLE: [memorable name]\n"
        f"CONCEPT: [2-3 sentences — what it is]\n"
        f"RATIONALE: [why this serves the updated brief]\n"
        f"EMOTIONAL TERRITORY: [how it feels]\n"
        f"RESEARCH_REQUEST: [optional — one specific question if needed]\n\n"
        f"These must be genuinely new directions, not refinements of the first set."
    )

    creator_output_2 = call_role(
        role="creator",
        user_message=creator_task_2,
        context=brief["challenge"],
        blackboard=blackboard,
        model=SESSION_MODEL,
        max_tokens=2400,
        trait_profile=trait_profiles["creator"],
        brief_doc=read_brief_doc(session_id),
    )

    # Store in second_exploration
    blackboard["second_exploration"]["additional_cycles"].append(
        {
            "cycle_number": len(blackboard["ideation_cycles"]) + 1,
            "creator_proposals": [{"raw_response": creator_output_2}],
            "critic_feedback": [],
            "synthesized_directions": [],
        }
    )

    update_brief_doc(
        session_id, "CREATOR", "DIRECTIONS — SECOND LOOP PASS 1", creator_output_2
    )
    scribe_log(
        blackboard,
        "CREATOR",
        "second_loop_ideation_complete",
        "Second loop ideation complete. New directions generated from PIL signal.",
    )

    print(creator_output_2)
    print()

    # ── Critic Pass 1 (Second Loop) ──────────────────────────────────────
    print("── CRITIC: SECOND LOOP ──────────────────────────────────────")

    critic_task_2 = (
        f"You are the Critic. The Creator has produced a fresh set of directions "
        f"in response to new signal from the person in the loop.\n\n"
        f"BRIEF CHALLENGE: {brief['challenge']}\n"
        f"PIL SIGNAL THAT TRIGGERED THIS CYCLE:\n{all_signals}\n\n"
        f"NEW DIRECTIONS TO EVALUATE:\n{creator_output_2}\n\n"
        f"Evaluate each direction:\n"
        f"IDEA_ID: [B1, B2, B3]\n"
        f"WHAT HOLDS: [what works and why]\n"
        f"WHAT DOESN'T: [specific weakness or risk]\n"
        f"REFINEMENT PATH: [one concrete improvement]\n"
        f"RESEARCH_REQUEST: [optional]\n\n"
        f"After all three, note any cross-direction patterns if present.\n"
        f"Close with a synthesis direction if one is warranted."
    )

    critic_output_2 = call_role(
        role="critic",
        user_message=critic_task_2,
        context=brief["challenge"],
        blackboard=blackboard,
        model=SESSION_MODEL,
        max_tokens=2400,
        trait_profile=trait_profiles["critic"],
        brief_doc=read_brief_doc(session_id),
    )

    blackboard["second_exploration"]["additional_cycles"][-1]["critic_feedback"].append(
        {"raw_response": critic_output_2}
    )
    update_brief_doc(
        session_id, "CRITIC", "EVALUATION — SECOND LOOP PASS 1", critic_output_2
    )
    scribe_log(
        blackboard,
        "CRITIC",
        "second_loop_evaluation_complete",
        "Second loop Critic evaluation complete.",
    )

    print(critic_output_2)
    print()

    # ── Researcher (Second Loop) ──────────────────────────────────────────
    print("── RESEARCHER: SECOND LOOP ──────────────────────────────────")

    # Parse any RESEARCH_REQUESTs from second loop Creator and Critic
    creator_requests_2 = re.findall(
        r"RESEARCH_REQUEST:\s*(.+?)(?=\n|$)", creator_output_2
    )
    critic_requests_2 = re.findall(
        r"RESEARCH_REQUEST:\s*(.+?)(?=\n|$)", critic_output_2
    )
    research_requests_2 = [
        f"[Creator]: {r.strip()}" for r in creator_requests_2 if r.strip()
    ] + [f"[Critic]:  {r.strip()}" for r in critic_requests_2 if r.strip()]

    if research_requests_2:
        research_task_2 = (
            f"You are supporting a creative studio session.\n\n"
            f"THE BRIEF:\n"
            f"Challenge: {brief['challenge']}\n"
            f"Context: {brief['context']}\n\n"
            f"SPECIFIC RESEARCH REQUESTS FROM THE CREATIVE TEAM:\n"
            + "\n".join(research_requests_2)
            + f"\n\nAddress each request with a named source, finding, and "
            f"relevance to the current directions. If you have an autonomous "
            f"contribution that would genuinely serve this cycle, include it "
            f"labeled UNSOLICITED CONTRIBUTION."
        )

        research_response_2 = call_role(
            role="researcher",
            user_message=research_task_2,
            context=brief["challenge"],
            blackboard=blackboard,
            model=SESSION_MODEL,
            max_tokens=2000,
            trait_profile=trait_profiles["researcher"],
            brief_doc=read_brief_doc(session_id),
        )

        blackboard["research_trace"].append(
            {
                "research_id": f"R{len(blackboard['research_trace']) + 1:03d}",
                "initiated_by": "creative_team_second_loop",
                "query": "Second loop research requests",
                "sources": ["researcher_knowledge"],
                "summary": research_response_2,
                "influence_on_ideation": "Second loop research delivered",
            }
        )
        update_brief_doc(
            session_id,
            "RESEARCHER",
            "RESEARCH FINDINGS — SECOND LOOP",
            research_response_2,
        )
        scribe_log(
            blackboard,
            "RESEARCHER",
            "second_loop_research_delivered",
            f"Second loop research delivered. "
            f"Entry R{len(blackboard['research_trace']):03d} logged.",
        )
        print(research_response_2)
    else:
        research_response_2 = ""
        print("  No research requests from second loop team.")

    print()

    # ── Update candidate set from second loop output ─────────────────────
    # Parse B-series direction IDs and titles from Critic synthesis
    # Director review will re-run in the synthesis cell using updated output
    import re as _re

    b_ids = _re.findall(r"IDEA_ID:\s*(B\d+)", creator_output_2)
    b_titles = _re.findall(r"TITLE:\s*(.+)", creator_output_2)

    # Replace candidate set with second loop directions
    blackboard["candidate_set"] = []
    for i, bid in enumerate(b_ids[:3]):
        blackboard["candidate_set"].append(
            {
                "direction_id": f"C{i+1}",
                "internal_type": "second_loop",
                "description": b_titles[i].strip() if i < len(b_titles) else bid,
                "reasoning_summary": "",
                "research_influences": [],
                "critic_assessment": {
                    "strengths": [],
                    "concerns": [],
                    "refinement_notes": [],
                },
            }
        )

    # Check for Critic synthesis direction
    if "SYNTHESIS_ID" in critic_output_2:
        s_titles = _re.findall(r"TITLE:\s*(.+)", critic_output_2)
        if s_titles:
            blackboard["candidate_set"].append(
                {
                    "direction_id": f"C{len(blackboard['candidate_set'])+1}",
                    "internal_type": "second_loop_synthesis",
                    "description": s_titles[0].strip(),
                    "reasoning_summary": "",
                    "research_influences": [],
                    "critic_assessment": {
                        "strengths": [],
                        "concerns": [],
                        "refinement_notes": [],
                    },
                }
            )

    # Update creator_output and critic_output for Cell 11 synthesis
    creator_output = creator_output_2
    critic_output = critic_output_2

    scribe_log(
        blackboard,
        "DIRECTOR",
        "second_loop_complete",
        f"Second loop complete. {len(blackboard['candidate_set'])} "
        f"new candidates. Proceeding to synthesis.",
    )

    print("── SECOND LOOP COMPLETE ─────────────────────────────────────")
    print(f"  New candidates     : {len(blackboard['candidate_set'])}")
    for c in blackboard["candidate_set"]:
        print(f"    {c['direction_id']} | {c['description'][:60]}")
    print(f"\n✓ creator_output and critic_output updated for synthesis")
    print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

---
## Cell 11 — Stage 9: Final Synthesis

Director produces the final synthesis — refining the chosen direction
and incorporating signals from the PIL's reaction.
This is the studio's best work.

In [ ]:
# ── STAGE 9: FINAL SYNTHESIS ─────────────────────────────────────────────────

print("═" * 60)
print("STAGE 9 — FINAL SYNTHESIS")
print("═" * 60)

all_signals = "\n\n".join(blackboard["user_response"]["signals_extracted"])
signals = all_signals if all_signals else ""

synthesis_task = f"""Produce the final synthesis for the PIL.

BRIEF: {brief['challenge']}
DESIRED RESULT: {brief['desired_result']}

PIL REACTION: {pil_reaction}

SIGNALS EXTRACTED:
{signals}

STUDIO DIRECTIONS EXPLORED:
{creator_output[:400]}

Synthesize the most compelling direction — refined by the PIL's signals.
This may combine elements from multiple directions.

Name this document appropriately for the type of problem solved. Use terms
such as: Action Plan, Creative Direction, Business Launch Plan, Brand Strategy,
Product Roadmap, or similar. Do not use "Final Synthesis" under any circumstances.

Begin with a single sentence introduction -- warm, direct, reflective of genuine
authorship by the full creative team, informed by everything the PIL shared.

Structure:
[DOCUMENT NAME]: [title]
[intro sentence]
CORE CONCEPT: [3-4 sentences]
WHY THIS WORKS: [2-3 sentences]
NEXT STEPS: [2-3 concrete actions]

Close with a single sentence inviting the PIL's reaction. Open and genuine.

Tone: confident, clear, earned."""

synthesis_response = call_role(
    role="director",
    user_message=synthesis_task,
    context=brief["challenge"],
    blackboard=blackboard,
    model=DIRECTOR_MODEL,
    max_tokens=800,
    brief_doc=read_brief_doc(session_id),
)

# Store final synthesis
synthesis = blackboard["final_synthesis"]
synthesis["summary"] = synthesis_response
if blackboard["candidate_set"]:
    synthesis["final_direction"] = blackboard["candidate_set"][-1]["direction_id"]
synthesis["refinements"].append(f"Refined from PIL signal: '{pil_reaction[:100]}'")

scribe_log(
    blackboard,
    "DIRECTOR",
    "synthesis_complete",
    "Final synthesis delivered to PIL. Session complete.",
)

print("── FINAL SYNTHESIS ─────────────────────────────────────────")
print(synthesis_response)
print("────────────────────────────────────────────────────────────")
print(f"\n✓ Final synthesis complete")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

---
## Cell 12 — Session Record: Blackboard Inspection + Save

Complete session record. Full reasoning trace visible.
Session saved to JSON — raw material for Phase 3 visualization.

In [ ]:
# ── SESSION RECORD ────────────────────────────────────────────────────────────

print("═" * 60)
print("SESSION RECORD")
print("═" * 60)
print(f"Session ID      : {blackboard['session_metadata']['session_id']}")
print(f"Prompt          : {blackboard['user_prompt']}")
print(f"Model           : {SESSION_MODEL}")
print()
print(f"Ideation cycles : {len(blackboard['ideation_cycles'])}")
print(f"Research entries: {len(blackboard['research_trace'])}")
print(f"Candidate dirs  : {len(blackboard['candidate_set'])}")
print(f"Reasoning steps : {len(blackboard['reasoning_trace'])}")
print()
print("── REASONING TRACE ─────────────────────────────────────────")
for entry in blackboard["reasoning_trace"]:
    print(
        f"  {entry['step']:>2} | {entry['role']:<12} | {entry['action']:<22} | {entry['summary'][:60]}"
    )

print()
print("── CANDIDATE DIRECTIONS ────────────────────────────────────")
for c in blackboard["candidate_set"]:
    print(f"  {c['direction_id']} | {c['internal_type']:<22} | {c['description']}")

# -- Assemble evaluation payload ---------------------------------------
# Packages problem_prompt, baseline, and final synthesis for evaluator.
assemble_evaluation_payload(blackboard, baseline_response)

# Save full session
saved_path = save_session(blackboard)
print()
print(f"✓ Full session saved to: {saved_path}")
print()
print("Phase 1 complete. Full studio workflow executed.")
print()
print("This JSON is Phase 3 input — the semantic trajectory data.")
print("Every reasoning_trace entry is an utterance in conceptual space.")